In [1]:
import resource
# Hard cap THIS kernel at 8 GB: a runaway cell dies with MemoryError instead of
# swap-thrashing the whole WSL VM (which killed it repeatedly on 2026-07-22).
# Lower-only: raising a finite hard limit is forbidden.
_cap = 8 * 1024**3
_soft, _hard = resource.getrlimit(resource.RLIMIT_AS)
_new_soft = _cap if _hard == resource.RLIM_INFINITY else min(_cap, _hard)
resource.setrlimit(resource.RLIMIT_AS, (_new_soft, _hard))

import json
import urllib.request
from pathlib import Path

import geopandas as gpd
import shapely
import pyogrio
import pandas as pd
import os

# ---- Config -----------------------------------------------------------
GAGES_PATH = "../tethysapp/hydro_correlation_tool/public/data/merged_gages.geojson"
BUCKET = "https://geoglows-v2.s3.amazonaws.com"
VPU_BOUNDS_URL = f"/vsicurl/{BUCKET}/hydrography-global/vpu-boundaries.gpkg"
HYDRO_DIR = Path("geoglows_hydrography")   # downloaded streams gpkgs land here (gitignored)
HYDRO_DIR.mkdir(exist_ok=True)

CRS_METERS = "EPSG:5070"   # Albers CONUS — buffer/snap distances in real meters
MIN_STREAM_ORDER = 2       # skip order-1 headwaters so gages don't snap to tiny tributaries
K = 5
R_INIT = 500
R_MAX = 50_000

In [2]:
# ---- Load gages -------------------------------------------------------
gages = gpd.read_file(GAGES_PATH)
print(len(gages), "gages |", gages.crs)
gages.head(3)

9113 gages | EPSG:4326


,USGSID,station_nm,COMID,geometry
0,USGS-02339495,OSELIGEE CREEK NEAR LANETT AL,3296804,POINT (-85.19633 32.90152)
1,USGS-02342500,"UCHEE CREEK NEAR FORT MITCHELL, AL.",3435970,POINT (-85.01493 32.31681)
2,USGS-02342937,"SOUTH FORK COWIKEE CREEK NR HOWE, AL",3437910,POINT (-85.2138 32.01679)


In [3]:
SEED_CSV = 'seed.csv'

In [4]:
seed = pd.read_csv(SEED_CSV, dtype={"geoglows_river_id": "Int64", "nwm_feature_id": "Int64"})
print (seed.dtypes)
print (len(seed))

usgs_id                 object
gage_name               object
latitude               float64
longitude              float64
nwm_feature_id           Int64
geoglows_river_id        Int64
verification_status     object
dtype: object
9113


## 1. Gage → VPU assignment (cached)

Same sjoin as seeding cells 4–5, but the result is saved to `gage_vpu.csv`
so the 1.9 GB `/vsicurl/` boundaries read only ever happens once.
The file's existence *is* the "already done" flag — it survives kernel restarts,
which a Python variable can't.

In [5]:
VPU_CACHE = Path("gage_vpu.csv")

if VPU_CACHE.exists():
    vpu_col = "VPU"
    cached = pd.read_csv(VPU_CACHE, dtype={"USGSID": str, "VPU": "Int64"})
    gage_vpu = gages.merge(cached, on="USGSID", how="left")
    print("loaded gage→VPU from cache")
else:
    conus_bbox = tuple(gages.total_bounds)  # (minx, miny, maxx, maxy) in 4326
    vpus = gpd.read_file(VPU_BOUNDS_URL, bbox=conus_bbox)   # slow: streams 1.9 GB gpkg over HTTP
    # The boundaries file mixes 2D/3D geometries and VPU 714's ring is invalid —
    # either one crashes batch to_crs(5070) on geopandas>=1.1. Flatten + repair first.
    vpus["geometry"] = shapely.make_valid(shapely.force_2d(vpus.geometry.values))
    vpu_col = next(c for c in vpus.columns if "vpu" in c.lower())
    vpus[vpu_col] = pd.to_numeric(vpus[vpu_col]).astype("Int64")  # codes ship as strings

    # A gage belongs to the VPU polygon that contains it.
    gage_vpu = gpd.sjoin(gages, vpus[[vpu_col, "geometry"]].to_crs(gages.crs),
                         how="left", predicate="within")

    # Coastal/border gages can fall just outside every polygon -> nearest VPU
    missing = gage_vpu[vpu_col].isna()
    if missing.any():
        near = gpd.sjoin_nearest(gages.loc[missing.values, ["USGSID", "geometry"]].to_crs(CRS_METERS),
                                 vpus[[vpu_col, "geometry"]].to_crs(CRS_METERS), how="left")
        gage_vpu.loc[missing, vpu_col] = near[vpu_col].values
        print(f"assigned {missing.sum()} stray gages to nearest VPU")

    gage_vpu[["USGSID", vpu_col]].to_csv(VPU_CACHE, index=False)
    print("computed and cached gage→VPU")

conus_vpus = sorted(gage_vpu[vpu_col].dropna().unique())
print("CONUS VPUs:", conus_vpus)
print("Gages per VPU:")
print(gage_vpu[vpu_col].value_counts(dropna=False))

loaded gage→VPU from cache
CONUS VPUs: [np.int64(701), np.int64(702), np.int64(703), np.int64(704), np.int64(706), np.int64(708), np.int64(709), np.int64(712), np.int64(713), np.int64(714), np.int64(715)]
Gages per VPU:
VPU
714    3060
712    1319
713    1219
703     868
704     792
715     689
709     597
702     438
706     114
708      14
701       3
Name: count, dtype: Int64


## 2. Nearest-K candidate reaches

**Border decision (2026-07-22):** only 83/9,113 gages sit within 1 km of their VPU
edge (438 within 5 km), and many of those "edges" are coastline. Rather than
special-case them, each tile below reads its stream window from **every** VPU file
it overlaps — the border problem disappears in the normal path. (`gage_vpu.csv`
keeps a per-gage `edge_m` column from that analysis, computed offline.)

**Memory design:** gages are processed in 150 km tiles; each tile loads only the
streams inside its window (+`R_MAX` margin, scaled for EPSG:3857 stretch), capped
to order ≥ `MIN_STREAM_ORDER`, id/order columns only. Peak footprint is a few
hundred MB by construction. Do NOT load whole VPU stream files in this notebook.

In [ ]:
def k_nearest_reaches(pt, streams, sindex, id_col, k=K, r0=R_INIT, r_max=R_MAX):
    """The k nearest reaches to one gage point, as [(reach_id, distance_m), ...].

    Expanding search: ask the spatial index for everything within radius r; if
    fewer than k reaches, double r and retry. Guard: accept only when the k-th
    exact distance <= r — everything not yet seen is farther than r, but hits
    near the search-square corners are only proven nearest once that holds.
    Distances greater than r_max (rare, sparse deserts) are best-effort.
    """
    r = r0
    while True:
        hits = sindex.query(pt.buffer(r))
        if len(hits) >= k or r >= r_max:
            d = shapely.distance(streams.geometry.values[hits], pt)
            order = d.argsort()[:k]
            if r >= r_max or (len(order) > 0 and d[order[-1]] <= r):
                ids = streams[id_col].values[hits][order]
                return [(int(i), float(x)) for i, x in zip(ids, d[order])]
        r *= 2

In [ ]:
CHUNK_M = 150_000                 # tile size for gage processing, meters (5070)
MARGIN_M = int(R_MAX * 1.6)       # window pad in 3857 units; x1.6 covers 3857's
                                  # 1/cos(lat) stretch so >= R_MAX real meters remain

def streams_path(vpu):
    return HYDRO_DIR / f"streams_{vpu}.gpkg"

# field names + extents read from file headers only (no feature data)
_info = pyogrio.read_info(streams_path(conus_vpus[0]))
id_col = next(f for f in _info["fields"] if f.lower() == "linkno")
order_col = next((f for f in _info["fields"] if "order" in f.lower()), None)
file_crs = _info["crs"]
print(f"id column {id_col!r} | order column {order_col!r} | file CRS {file_crs}")

stream_files = [(p, pyogrio.read_info(p)["total_bounds"]) for p in sorted(HYDRO_DIR.glob("streams_*.gpkg"))]
print(len(stream_files), "stream files on disk")

gages_m = gage_vpu[["USGSID", "geometry"]].to_crs(CRS_METERS)
seed_map = (seed.dropna(subset=["geoglows_river_id"])
                .set_index("usgs_id")["geoglows_river_id"].astype("int64").to_dict())

matches = {}                      # USGSID -> [(reach_id, dist_m), ...]
seed_missing_from_window = []
done = 0

tx = (gages_m.geometry.x // CHUNK_M).astype(int)
ty = (gages_m.geometry.y // CHUNK_M).astype(int)

for (i, j), tile in gages_m.groupby([tx, ty]):
    w = tile.to_crs(file_crs).total_bounds
    window = (w[0] - MARGIN_M, w[1] - MARGIN_M, w[2] + MARGIN_M, w[3] + MARGIN_M)

    parts = []
    for path, (fx0, fy0, fx1, fy1) in stream_files:
        if fx1 < window[0] or fx0 > window[2] or fy1 < window[1] or fy0 > window[3]:
            continue                      # file nowhere near this tile
        parts.append(gpd.read_file(
            path, bbox=window,
            columns=[id_col] + ([order_col] if order_col else []),
            where=f'"{order_col}" >= {MIN_STREAM_ORDER}' if order_col else None))
    parts = [p for p in parts if not p.empty]
    if not parts:
        for usgs_id in tile["USGSID"]:
            matches[usgs_id] = []
        continue

    streams = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True),
                               geometry="geometry", crs=file_crs).to_crs(CRS_METERS)
    sindex = streams.sindex
    id_pos = pd.Index(streams[id_col])

    for usgs_id, pt in zip(tile["USGSID"], tile.geometry):
        cands = k_nearest_reaches(pt, streams, sindex, id_col)
        sid = seed_map.get(usgs_id)
        if sid is not None and sid not in {c for c, _ in cands}:   # force-include the seeded id
            pos = id_pos.get_indexer([sid])[0]
            if pos == -1:
                seed_missing_from_window.append(usgs_id)
            else:
                cands.append((int(sid), float(shapely.distance(streams.geometry.values[pos], pt))))
        matches[usgs_id] = cands

    done += len(tile)
    print(f"tile ({i},{j}): {len(tile):4d} gages | {len(streams):6d} reaches in window | {done}/{len(gages_m)}", flush=True)

print("search complete:", len(matches), "gages |",
      len(seed_missing_from_window), "seeded ids not found in any window")

In [ ]:
rows = []
for usgs_id, cands in matches.items():
    for rank, (rid, dm) in enumerate(sorted(cands, key=lambda t: t[1]), start=1):
        rows.append((usgs_id, "geoglows", rid, rank, round(dm, 1)))
cand_df = pd.DataFrame(rows, columns=["usgs_id", "network", "reach_id", "rank", "distance_m"])

# ---- QA ----------------------------------------------------------------
n_per = cand_df.groupby("usgs_id").size()
print(len(cand_df), "candidate rows |", cand_df["usgs_id"].nunique(), "gages")
print("candidates per gage:"); print(n_per.value_counts().sort_index())
print("gages with fewer than", K, "candidates:", (n_per < K).sum())

print("rank-1 distances (should echo the seeding snap-distance histogram):")
print(cand_df.loc[cand_df["rank"] == 1, "distance_m"].describe())

seeded = seed[["usgs_id", "geoglows_river_id"]].dropna()
hit = (cand_df.merge(seeded, on="usgs_id")
              .assign(hit=lambda d: d["reach_id"] == d["geoglows_river_id"])
              .groupby("usgs_id")["hit"].any())
print("seeded gages whose seeded id is in their list:", int(hit.sum()), "/", len(hit))

cand_df.to_csv("candidates_geoglows.csv", index=False)
print("wrote candidates_geoglows.csv")